# rerank_topicality_feature — the orthogonal topicality signal (§9e #2)

Asks Qwen "is this trial *about* the patient's condition?" over the topicality fields (conditions/title/
summary), deliberately orthogonal to the eligibility judge. The diagnostic showed the ensemble can stack
an orthogonal LLM signal (the eligibility judge was the strongest feature) — this adds a second, uncorrelated
one to recover the diversity lost when clf_R/reranker_v3 became redundant on R. Writes `topicality_R.jsonl`.


## Setup (Colab — GPU, ~14GB for Qwen-7B; ~40 min like the judge)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, llm_yesno_scores,
                                 llm_topicality_prompt, ndcg_at_k)
POOL_TAG = 'R'   # set 'nqs' to re-score on the NQS pool
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)
print('topicality fields:', cfg.topicality_fields, '| judge:', cfg.llm_ckpt)


In [ ]:
# Same retrieval pool (top llm_top_k) as the eligibility judge, so the two features align per (topic, doc).
SETS = ['trec21', 'kz', 'trec22']
OUT_PATH = cfg.feat_file('topicality')
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, SETS)
rp = json.load(open(cfg.pool_path()))
pools = {s: {t: [d for d in docs[:cfg.llm_top_k] if d in id2fields] for t, docs in rp[s].items()} for s in SETS}
print({s: sum(len(v) for v in p.values()) for s, p in pools.items()}, 'pairs')


In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.llm_ckpt, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(cfg.llm_ckpt, torch_dtype=torch.float16, device_map='auto').eval()
print('judge loaded')


In [ ]:
# Score topicality (prompt_fn = llm_topicality_prompt) over the pool; write the feature jsonl.
with open(OUT_PATH, 'w') as out:
    for s in SETS:
        for t, docs in tqdm(pools[s].items(), desc=f'topicality {s}'):
            scores = llm_yesno_scores(llm, tok, sets[s]['topic2text'][t], [id2fields[d] for d in docs], cfg,
                                      batch=8, prompt_fn=llm_topicality_prompt)
            for d, sc in zip(docs, scores):
                out.write(json.dumps({'source': s, 'topic_id': t, 'doc_id': d, 'topicality': float(sc)}) + '\n')
print('wrote', OUT_PATH)


In [ ]:
# Sanity: standalone topicality NDCG@10, and how correlated it is with the eligibility judge.
topi = {}; elig = {}
for l in open(OUT_PATH):
    r = json.loads(l); topi[(r['source'], r['topic_id'], r['doc_id'])] = r['topicality']
for l in open(cfg.feat_file('llm_scores')):
    r = json.loads(l); elig[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']; vals = []
    for t, docs in pools[s].items():
        ranked = sorted(docs, key=lambda d: topi[(s,t,d)], reverse=True)
        vals.append(ndcg_at_k(ranked, rel[t]))
    rows.append({'split': s, 'topicality_ndcg@10': round(float(np.mean(vals)), 4)})
common = [k for k in topi if k in elig]
corr = np.corrcoef([topi[k] for k in common], [elig[k] for k in common])[0,1]
print('corr(topicality, eligibility judge) =', round(float(corr), 3), '(low = orthogonal = good)')
pd.DataFrame(rows)


## Reading it
The point isn't standalone NDCG — it's **low correlation with the eligibility judge** (and with clf_R).
A topicality feature that's ~orthogonal is one the LambdaMART can stack. Then re-run `train_ensemble_full`
(it auto-includes `topicality_R.jsonl` as a 10th feature) to see if it recovers the diversity toward 0.61.
